In [2]:
"""
Project: Whisper Speech-to-Text (STT) Implementation
Author: Armin Seyfi
Date: 09.10.2026

Description:
    Implements an audio transcription workflow using OpenAI's Whisper API.
    Processes meeting recordings, handles long audio through chunking,
    extracts segment-level timestamps, and compares guided (prompted)
    and unguided (unprompted) transcription approaches.

Business Use Case:
    Convert business meeting recordings into accurate, timestamped,
    searchable transcripts for documentation and information retrieval.

Input:
    NY045.mp3 (located in the project's root directory).

Output:
    - Transcribed speech in text format.
    - Timestamped transcription segments.
    - Comparison of prompted and unprompted transcription results.
    - Exportable transcripts in readable formats.
"""

"\nProject: Whisper Speech-to-Text (STT) Implementation\nAuthor: Armin Seyfi\nDate: 09.10.2026\n\nDescription:\n    Implements an audio transcription workflow using OpenAI's Whisper API.\n    Processes meeting recordings, handles long audio through chunking,\n    extracts segment-level timestamps, and compares guided (prompted)\n    and unguided (unprompted) transcription approaches.\n\nBusiness Use Case:\n    Convert business meeting recordings into accurate, timestamped,\n    searchable transcripts for documentation and information retrieval.\n\nInput:\n    NY045.mp3 (located in the project's root directory).\n\nOutput:\n    - Transcribed speech in text format.\n    - Timestamped transcription segments.\n    - Comparison of prompted and unprompted transcription results.\n    - Exportable transcripts in readable formats.\n"

In [6]:
pip install openai pydub python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load API credentials
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Open and transcribe audio
with open("NY045.mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

# Display transcription
print(transcription.text)

Oh, going to Cuba. I went to Cuba on Prohibition time too, and I bought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning. And, oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles. And, oh, the captain says, get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba. And the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag. So they found a $50 and threw it over the side of the boat. So then I got panicky because I had about six little bottles, and I thought, I'll hav

In [8]:
# Step 4: Guided Transcription with Context Prompt

# Define domain-specific context
context_prompt = (
    "A historical oral-history recording about traveling to Cuba "
    "during the Prohibition era, bringing liquor bottles aboard "
    "a ship, customs inspections, Atlantic City, and souvenirs. "
    "The speaker uses informal conversational American English."
)

# Transcribe audio using contextual guidance
with open("NY045.mp3", "rb") as audio_file:
    guided_transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=context_prompt
    )

# Display the result
print(guided_transcription.text)

Oh, going to Cuba. I went to Cuba in Prohibition time, too, and I brought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it, too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning, and oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles, and oh, the captain says, Get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba, and the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag, so they found a $50 and threw it over the side of the boat. So then I got panicky because I had about six little bottles, and I thought, I'll ha

In [10]:
# Compare unguided and guided transcription results

unguided_text = transcription.text
guided_text = guided_transcription.text

print("=== UNGUIDED TRANSCRIPTION ===")
print(unguided_text)

print("\n=== GUIDED TRANSCRIPTION ===")
print(guided_text)

=== UNGUIDED TRANSCRIPTION ===
Oh, going to Cuba. I went to Cuba on Prohibition time too, and I bought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning. And, oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles. And, oh, the captain says, get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba. And the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag. So they found a $50 and threw it over the side of the boat. So then I got panicky because I had about six little b

In [11]:
import difflib

# Compare the two transcription outputs
diff = difflib.unified_diff(
    unguided_text.splitlines(),
    guided_text.splitlines(),
    fromfile="Unguided",
    tofile="Guided",
    lineterm=""
)

print("\n".join(diff))

--- Unguided
+++ Guided
@@ -1 +1 @@
-Oh, going to Cuba. I went to Cuba on Prohibition time too, and I bought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning. And, oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles. And, oh, the captain says, get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba. And the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag. So they found a $50 and threw it over the side of the boat. So then I got panicky because I had about six li

In [12]:
# Save transcription results

with open("transcription_unguided.txt", "w", encoding="utf-8") as f:
    f.write(unguided_text)

with open("transcription_guided.txt", "w", encoding="utf-8") as f:
    f.write(guided_text)

print("Both transcription files saved successfully.")

Both transcription files saved successfully.


In [13]:
%pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Load the human-verified reference transcript

with open("NY045_reference.txt", "r", encoding="utf-8") as f:
    reference_text = f.read()

In [15]:
import re
import string

def normalize_transcript(text):
    # Remove speaker labels
    text = re.sub(r"\[(INF|FW):\]", "", text)

    # Convert to lowercase
    text = text.lower()

    # Normalize apostrophes
    text = text.replace("’", "'").replace("‘", "'")

    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    # Normalize whitespace
    text = " ".join(text.split())

    return text

In [16]:
from jiwer import wer, cer
import pandas as pd

# Normalize all transcripts
reference = normalize_transcript(reference_text)
unguided = normalize_transcript(transcription.text)
guided = normalize_transcript(guided_transcription.text)

# Calculate evaluation metrics
results = pd.DataFrame({
    "Approach": ["Unguided", "Guided"],
    "WER": [
        wer(reference, unguided),
        wer(reference, guided)
    ],
    "CER": [
        cer(reference, unguided),
        cer(reference, guided)
    ]
})

# Convert metrics to percentages
results["WER (%)"] = (results["WER"] * 100).round(2)
results["CER (%)"] = (results["CER"] * 100).round(2)

display(results[["Approach", "WER (%)", "CER (%)"]])

,Approach,WER (%),CER (%)
0,Unguided,15.63,7.83
1,Guided,15.63,7.92


In [17]:
from jiwer import process_words

# Analyze word-level errors
unguided_errors = process_words(reference, unguided)
guided_errors = process_words(reference, guided)

error_comparison = pd.DataFrame({
    "Metric": ["Substitutions", "Deletions", "Insertions", "WER"],
    "Unguided": [
        unguided_errors.substitutions,
        unguided_errors.deletions,
        unguided_errors.insertions,
        round(unguided_errors.wer, 4)
    ],
    "Guided": [
        guided_errors.substitutions,
        guided_errors.deletions,
        guided_errors.insertions,
        round(guided_errors.wer, 4)
    ]
})

display(error_comparison)

,Metric,Unguided,Guided
0,Substitutions,44.0000,43.0000
1,Deletions,22.0000,23.0000
2,Insertions,7.0000,7.0000
3,WER,0.1563,0.1563


In [18]:
from pydub import AudioSegment
from pathlib import Path

# Load original audio
audio = AudioSegment.from_mp3("NY045.mp3")

# Define chunk duration (60 seconds)
chunk_length_ms = 60 * 1000

# Create output directory
output_dir = Path("audio_chunks")
output_dir.mkdir(exist_ok=True)

# Split and export audio chunks
chunk_files = []

for i, start_ms in enumerate(range(0, len(audio), chunk_length_ms)):

    end_ms = min(start_ms + chunk_length_ms, len(audio))

    chunk = audio[start_ms:end_ms]

    chunk_path = output_dir / f"chunk_{i+1:02d}.mp3"

    chunk.export(chunk_path, format="mp3")

    chunk_files.append({
        "file": str(chunk_path),
        "start_seconds": start_ms / 1000,
        "end_seconds": end_ms / 1000
    })

    print(
        f"Chunk {i+1}: "
        f"{start_ms/1000:.1f}s - {end_ms/1000:.1f}s"
    )

print(f"\nTotal chunks created: {len(chunk_files)}")

Chunk 1: 0.0s - 60.0s
Chunk 2: 60.0s - 120.0s
Chunk 3: 120.0s - 180.0s
Chunk 4: 180.0s - 203.6s

Total chunks created: 4


In [19]:
# Verify chunking results

original_duration = len(audio) / 1000

total_chunk_duration = sum(
    item["end_seconds"] - item["start_seconds"]
    for item in chunk_files
)

print(f"Original duration: {original_duration:.2f} seconds")
print(f"Total chunk duration: {total_chunk_duration:.2f} seconds")

assert abs(original_duration - total_chunk_duration) < 0.001

print("Chunking verification passed!")

Original duration: 203.62 seconds
Total chunk duration: 203.62 seconds
Chunking verification passed!


In [20]:
# Transcribe individual audio chunks

chunk_transcriptions = []

for chunk in chunk_files:

    print(f"Processing: {chunk['file']}")

    with open(chunk["file"], "rb") as audio_file:

        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )

    chunk_transcriptions.append({
        "start_seconds": chunk["start_seconds"],
        "end_seconds": chunk["end_seconds"],
        "text": response.text
    })

print("\nAll chunks transcribed successfully!")

Processing: audio_chunks/chunk_01.mp3
Processing: audio_chunks/chunk_02.mp3
Processing: audio_chunks/chunk_03.mp3
Processing: audio_chunks/chunk_04.mp3

All chunks transcribed successfully!


In [21]:
# Combine transcriptions with chunk-level timestamps

combined_transcript = []

for chunk in chunk_transcriptions:

    start = chunk["start_seconds"]
    end = chunk["end_seconds"]

    combined_transcript.append(
        f"[{start:.0f}s - {end:.0f}s] {chunk['text']}"
    )

final_transcript = "\n\n".join(combined_transcript)

print(final_transcript)

[0s - 60s] Oh, going to Cuba. I went to Cuba on Prohibition time, too, and I bought a lot of fancy bottles, a little, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it, too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning, and oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles. And oh, the captain says, Get rid of that stuff. They're searching the boats. They're downstairs ripping the wall upon. So, and...

[60s - 120s] They looked us all over for liquor and for bringing things in from Cuba, and the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag, so they found her $50 and threw it over the side of the boat. So then I got panicky because I had about

In [22]:
with open("NY045_chunked_transcript.txt", "w", encoding="utf-8") as f:
    f.write(final_transcript)

print("Chunked transcript saved successfully.")

Chunked transcript saved successfully.


In [23]:
response_format="verbose_json"
timestamp_granularities=["segment"]

In [24]:
# Step 7A: Transcribe chunks with segment timestamps

timestamped_chunks = []

for chunk in chunk_files:

    print(f"Processing: {chunk['file']}")

    with open(chunk["file"], "rb") as audio_file:

        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    timestamped_chunks.append({
        "file": chunk["file"],
        "offset": chunk["start_seconds"],
        "segments": response.segments
    })

print("\nAll chunks transcribed with timestamps!")

Processing: audio_chunks/chunk_01.mp3
Processing: audio_chunks/chunk_02.mp3
Processing: audio_chunks/chunk_03.mp3
Processing: audio_chunks/chunk_04.mp3

All chunks transcribed with timestamps!


In [25]:
# Inspect the first transcription segment

first_segment = timestamped_chunks[0]["segments"][0]

print(first_segment.model_dump())

{'id': 0, 'avg_logprob': -0.3425324559211731, 'compression_ratio': 1.4473683834075928, 'end': 14.0, 'no_speech_prob': 0.035117778927087784, 'seek': 0, 'start': 0.0, 'temperature': 0.0, 'text': ' Oh, going to Cuba. I went to Cuba on Prohibition time, too, and I bought a lot of fancy bottles,', 'tokens': [50364, 876, 11, 516, 281, 20604, 13, 286, 1437, 281, 20604, 322, 1705, 5455, 849, 565, 11, 886, 11, 293, 286, 4243, 257, 688, 295, 10247, 15923, 11, 51064]}


In [26]:
# Step 7C: Adjust timestamps to original audio timeline

all_segments = []

for chunk in timestamped_chunks:

    offset = chunk["offset"]

    for segment in chunk["segments"]:

        global_start = segment.start + offset
        global_end = segment.end + offset

        all_segments.append({
            "start": global_start,
            "end": global_end,
            "text": segment.text.strip()
        })

# Sort segments chronologically
all_segments.sort(key=lambda x: x["start"])

print(f"Total segments: {len(all_segments)}")

Total segments: 30


In [27]:
# Convert seconds to HH:MM:SS format

def format_timestamp(seconds):

    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)

    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

In [28]:
print(format_timestamp(135))

00:02:15


In [29]:
# Step 7E: Generate timestamped transcript

formatted_segments = []

for segment in all_segments:

    start = format_timestamp(segment["start"])
    end = format_timestamp(segment["end"])

    formatted_segments.append(
        f"[{start} --> {end}] {segment['text']}"
    )

final_timestamped_transcript = "\n".join(formatted_segments)

print(final_timestamped_transcript)

[00:00:00 --> 00:00:14] Oh, going to Cuba. I went to Cuba on Prohibition time, too, and I bought a lot of fancy bottles,
[00:00:14 --> 00:00:21] a little, like one drink in for souvenirs for all my friends. Well, Atlantic City, the
[00:00:21 --> 00:00:29] ship was stopped. It was all in the newspapers about it, too. The crew had ripped the walls
[00:00:29 --> 00:00:37] and put all liquor in the walls. So we all had to get up at three o'clock in the morning,
[00:00:37 --> 00:00:42] and oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they
[00:00:42 --> 00:00:48] had bottles sticking all out all over, and anyone could see they had the bottles. And
[00:00:48 --> 00:00:54] oh, the captain says, Get rid of that stuff. They're searching the boats. They're downstairs
[00:00:54 --> 00:00:59] ripping the wall upon. So, and...
[00:01:00 --> 00:01:10] They looked us all over for liquor and for bringing things in from Cuba,
[00:01:10 --> 00:01:16] and the girl in f

In [30]:
# Step 7F: Export timestamped transcript

output_file = "NY045_timestamped_transcript.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_timestamped_transcript)

print(f"Transcript exported successfully: {output_file}")

Transcript exported successfully: NY045_timestamped_transcript.txt


In [31]:
# Step 7G: Validate timestamp boundaries

audio_duration = len(audio) / 1000

invalid_segments = []

for segment in all_segments:

    if (
        segment["start"] < 0
        or segment["end"] < segment["start"]
        or segment["end"] > audio_duration + 0.5
    ):
        invalid_segments.append(segment)

print(f"Audio duration: {audio_duration:.2f} seconds")
print(f"Total segments: {len(all_segments)}")
print(f"Invalid segments: {len(invalid_segments)}")

if not invalid_segments:
    print("Timestamp validation passed!")
else:
    print("Warning: Some timestamps are outside valid boundaries.")

Audio duration: 203.62 seconds
Total segments: 30
Invalid segments: 0
Timestamp validation passed!


In [32]:
from pathlib import Path
import json

# Create export directory
export_dir = Path("transcription_exports")
export_dir.mkdir(exist_ok=True)

print(f"Export directory ready: {export_dir}")

Export directory ready: transcription_exports


In [33]:
# Export human-readable transcript to TXT

txt_path = export_dir / "NY045_transcript.txt"

with open(txt_path, "w", encoding="utf-8") as f:

    f.write("NY045 - Timestamped Transcription\n")
    f.write("=" * 40 + "\n\n")

    for segment in all_segments:

        start = format_timestamp(segment["start"])
        end = format_timestamp(segment["end"])

        f.write(f"[{start} --> {end}]\n")
        f.write(f"{segment['text']}\n\n")

print(f"TXT exported: {txt_path}")

TXT exported: transcription_exports/NY045_transcript.txt


In [34]:
# Convert seconds to SRT timestamp format

def format_srt_timestamp(seconds):

    total_ms = round(seconds * 1000)

    hours, remainder = divmod(total_ms, 3600000)
    minutes, remainder = divmod(remainder, 60000)
    secs, milliseconds = divmod(remainder, 1000)

    return (
        f"{hours:02d}:{minutes:02d}:"
        f"{secs:02d},{milliseconds:03d}"
    )

In [35]:
# Export subtitles to SRT

srt_path = export_dir / "NY045_subtitles.srt"

with open(srt_path, "w", encoding="utf-8") as f:

    for index, segment in enumerate(all_segments, start=1):

        start = format_srt_timestamp(segment["start"])
        end = format_srt_timestamp(segment["end"])

        f.write(f"{index}\n")
        f.write(f"{start} --> {end}\n")
        f.write(f"{segment['text']}\n\n")

print(f"SRT exported: {srt_path}")

SRT exported: transcription_exports/NY045_subtitles.srt


In [36]:
# Export structured transcription data to JSON

json_path = export_dir / "NY045_transcript.json"

transcription_data = {
    "source_file": "NY045.mp3",
    "model": "whisper-1",
    "duration_seconds": len(audio) / 1000,
    "total_segments": len(all_segments),
    "segments": all_segments
}

with open(json_path, "w", encoding="utf-8") as f:

    json.dump(
        transcription_data,
        f,
        indent=4,
        ensure_ascii=False
    )

print(f"JSON exported: {json_path}")

JSON exported: transcription_exports/NY045_transcript.json


In [37]:
# Verify exported files

exported_files = [txt_path, srt_path, json_path]

for file_path in exported_files:

    assert file_path.exists(), f"Missing file: {file_path}"
    assert file_path.stat().st_size > 0, f"Empty file: {file_path}"

    print(
        f"{file_path.name}: "
        f"{file_path.stat().st_size} bytes - OK"
    )

# Validate JSON structure
with open(json_path, "r", encoding="utf-8") as f:
    loaded_json = json.load(f)

assert len(loaded_json["segments"]) == len(all_segments)

# Validate SRT block structure
with open(srt_path, "r", encoding="utf-8") as f:
    srt_content = f.read().strip()

srt_blocks = srt_content.split("\n\n")

assert len(srt_blocks) == len(all_segments)

for index, block in enumerate(srt_blocks, start=1):
    lines = block.splitlines()
    assert lines[0] == str(index)
    assert " --> " in lines[1]
    assert len(lines) >= 3

print("\nAll export validations passed!")

NY045_transcript.txt: 3006 bytes - OK
NY045_subtitles.srt: 3191 bytes - OK
NY045_transcript.json: 5672 bytes - OK

All export validations passed!
